In [ ]:
# ==============================================================================
# 1. SETUP & IMPORTS
# ==============================================================================
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

print(f"Using TensorFlow version: {tf.__version__}")

# ==============================================================================
# 2. LOAD & PREPROCESS MNIST DATASET
# ==============================================================================
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize pixel values to [0.0, 1.0] float32
x_train = (x_train / 255.0).astype(np.float32)
x_test = (x_test / 255.0).astype(np.float32)

# Reshape inputs to include channel dimension: (Batch, 28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

# ==============================================================================
# 3. DEFINE CNN MODEL (Matches MicroPython C Architecture)
# Architecture:
# Input: (28, 28, 1)
# Conv2D: 4 filters, 3x3 kernel, stride=1, valid padding -> Output: (26, 26, 4)
# ReLU -> MaxPooling2D (2x2, stride=2) -> Output: (13, 13, 4)
# Flatten: 13 * 13 * 4 = 676
# Dense(16) -> ReLU
# Dense(10)
# ==============================================================================
model = models.Sequential([
    layers.Conv2D(4, (3, 3), strides=(1, 1), padding='valid', activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),
    layers.Flatten(),
    layers.Dense(16, activation='relu'),
    layers.Dense(10)  # Logits output (matches C soft/linear output)
])

model.summary()

# ==============================================================================
# 4. COMPILE & TRAIN MODEL
# ==============================================================================
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

print("\nStarting training...")
model.fit(x_train, y_train, epochs=10, batch_size=64, validation_data=(x_test, y_test))

# ==============================================================================
# 5. EXPORT WEIGHTS TO BINARY (.BIN) FILE FOR MICROPYTHON
# ==============================================================================
def export_tf_to_microml_bin(model, output_filepath="cnn_mnist.bin"):
    conv_layer = model.layers[0]   # Conv2D
    fc1_layer  = model.layers[3]   # Dense 16
    fc2_layer  = model.layers[4]   # Dense 10

    # 1. Conv2D Weights (Transposed to C Out_Ch, In_Ch, H, W layout)
    conv_w, conv_b = conv_layer.get_weights()
    conv_w_c_layout = np.transpose(conv_w, (3, 2, 0, 1)).astype(np.float32)
    conv_b_c_layout = conv_b.astype(np.float32)

    # 2. Dense Layer 1 (Fix Flattening Permutation mismatch)
    fc1_w, fc1_b = fc1_layer.get_weights() # Shape: (676, 16)

    # Reshape (13, 13, 4, 16) -> Transpose to (4, 13, 13, 16) -> Reshape back to (676, 16)
    fc1_w_reshaped = fc1_w.reshape(13, 13, 4, 16)
    fc1_w_c_layout = np.transpose(fc1_w_reshaped, (2, 0, 1, 3)).reshape(676, 16).astype(np.float32)
    fc1_b_c_layout = fc1_b.astype(np.float32)

    # 3. Dense Layer 2
    fc2_w, fc2_b = fc2_layer.get_weights()
    fc2_w_c_layout = fc2_w.astype(np.float32)
    fc2_b_c_layout = fc2_b.astype(np.float32)

    with open(output_filepath, "wb") as f:
        f.write(conv_w_c_layout.tobytes())
        f.write(conv_b_c_layout.tobytes())
        f.write(fc1_w_c_layout.tobytes())
        f.write(fc1_b_c_layout.tobytes())
        f.write(fc2_w_c_layout.tobytes())
        f.write(fc2_b_c_layout.tobytes())

    print(f"Exported aligned model weights to '{output_filepath}'!")

export_tf_to_microml_bin(model, "cnn_mnist.bin")



Using TensorFlow version: 2.20.0


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 4)      │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 13, 13, 4)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 676)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │        10,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │           170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,042 (43.13 KB)

 Trainable params: 11,042 (43.13 KB)

 Non-trainable params: 0 (0.00 B)


Starting training...
Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 23s 22ms/step - accuracy: 0.8717 - loss: 0.4468 - val_accuracy: 0.9371 - val_loss: 0.2173
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 17s 18ms/step - accuracy: 0.9455 - loss: 0.1873 - val_accuracy: 0.9562 - val_loss: 0.1459
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 17s 18ms/step - accuracy: 0.9598 - loss: 0.1385 - val_accuracy: 0.9620 - val_loss: 0.1217
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 19s 20ms/step - accuracy: 0.9664 - loss: 0.1148 - val_accuracy: 0.9680 - val_loss: 0.1003
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - accuracy: 0.9689 - loss: 0.1035 - val_accuracy: 0.9710 - val_loss: 0.0925
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 17s 18ms/step - accuracy: 0.9715 - loss: 0.0937 - val_accuracy: 0.9726 - val_loss: 0.0864
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 21s 19ms/step - accuracy: 0.9743 - loss: 0.0866 - val_accuracy: 0.9724 - val_loss: 0.0862
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 18s 19ms/step - accuracy: 0.